# Exercise 2D: Tethys-Chloris (T&C) spatially-distributed modelling

This exercise extends the plot-scale workflows of Exercises 1 and 2 to the **spatially-distributed (2D)** version of the T&C model. You will set up, run and analyse a small catchment at the Chamau grassland site (CH-Cha, Switzerland), exactly as in the original MATLAB demo (`GO_TeC_Demo.m`), but entirely in Julia.

**Objectives**

- Understand how a 2D (grid-based) T&C setup differs from the plot-scale (0D) version: terrain, routing and per-cell inputs.
- Create the model inputs from raw MATLAB data files.
- Run a short smoke simulation to validate the setup, then the full 5-month simulation.
- Analyse spatial maps and catchment-average time series.

**Case study**: the `GeoExample` catchment (50x50 cells of 40 m x 40 m) at Chamau, with two vegetation classes (wood and grass/shrub) and hourly forcing from the CH-Cha eddy-covariance station from 1 May to 3 October 2005.

## 1. Load the Julia environment

Exercise 2D uses the same Julia environment as Exercises 1 and 2 (the repository root `Project.toml`).

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

In [ ]:
using NCDatasets
using YAML
using Roots
using Plots
using TethysChloris

FT = Float64
data_dir = joinpath(@__DIR__, "data")
nothing

## 2. Understand the case study

The 2D model needs two kinds of input:

1. **Grid (geodata)**: digital terrain model, slope, aspect, flow routing matrix, stream network, land-use codes and soil texture for every cell.
2. **Forcing (meteorology)**: the same hourly station data as in Exercise 1 (precipitation, temperature, wind, humidity, radiation), distributed over the grid with lapse rates.

Before running anything, answer:

**Question 2.1**: Why does a 2D model need a *flow routing matrix* (`T_flow`) in addition to elevation and slope? What physical process does it control that the plot-scale model cannot represent?

**Question 2.2**: The catchment contains two land-use classes. Where do you expect surface runoff to differ most between the two, and why?

## 3. Create the model inputs

The script `create_data_2d.jl` reads the raw MATLAB files in `data/` (`Data_CH-Cha_run.mat`, `GeoExample.mat`, `Ca_Data.mat`) and writes:

- `data/exercise_2d_data.nc`: the NetCDF forcing + grid file
- `data/exercise_2d_parameters.yaml`: the parameter file for both land-use classes

Run it (this takes a few seconds):

In [ ]:
include(joinpath(@__DIR__, "create_data_2d.jl"))

### 3.1 Fill in the bedrock conductivity

In the MATLAB exercise, students must set `Kbot = [NaN NaN]` in `PARAMETERS_ALL_demo.m`. Here the generated YAML file contains the same gap. Open `data/exercise_2d_parameters.yaml` and find the `rock:` block (two occurrences, one per land-use class):

```yaml
rock:
  Kbot: .nan
  Krock: 0.1
  In_max_rock: 2.0
```

`Kbot` is the **vertical conductivity at the bedrock interface** [mm/h]. It controls how much water can leave the soil column downward into the fractured bedrock aquifer.

**Question 3.1**: Before looking below, write down your expectation: for a grassland catchment on shallow soil over impermeable bedrock, should `Kbot` be large or small compared to the saturated conductivity of the soil (here `Ks` of order tens of mm/h)?

**Question 3.2**: Set **both** occurrences of `Kbot` to `0.11` (the value used in the MATLAB demo: a nearly impermeable bedrock, 0.11 mm/h). Why would a modeller choose a small but *non-zero* value instead of exactly zero?

## 4. Smoke run: validate the setup

Before committing to the full simulation, run only 10 timesteps. This validates the data files, the parameter file (including your `Kbot` edit) and the model options in well under a minute.

The model options match the MATLAB demo and Exercises 1-2: soil temperature enabled, freezing soil processes enabled, vegetation-snow interactions enabled.

In [ ]:
netcdfpath = joinpath(data_dir, "exercise_2d_data.nc")
yamlpath = joinpath(data_dir, "exercise_2d_parameters.yaml")

model, forcing = initialize_model(Model2D{FT}, netcdfpath, yamlpath);

In [ ]:
options = ModelOptions(
    SoilBiogeochemistry = false,
    SoilTemperature = true,
    FreezingSoil = true,
    VegetationSnowInteractions = true,
    OPT_CR = RootsNonBracketingStrategy(FT),
    OPT_ST = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_ST2 = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_VD = ODEOptions(; abstol = 0.05),
);

In [ ]:
diagnostics, manager = run_simulation(
    model,
    forcing;
    options = options,
    run_options = SpatialRunOptions(
        NN = 10,
        output_level = AllOutputs(),
        output_filename = joinpath(data_dir, "smoke_results.nc"),
        compress = true,
        parallel = false,
        tstore = [10],
    ),
);

**Question 4.1**: The smoke run saved a NetCDF with several groups (`point_1`, `spatial_mean`, `spatial_std`, `time_average`, `grid`). Inspect it below: what is the shape of a `time_average` variable, and what does the last dimension mean?

In [ ]:
smoke = NCDataset(joinpath(data_dir, "smoke_results.nc"))
@show collect(keys(smoke.group))
@show size(smoke.group["time_average"]["Ts"])
@show size(smoke.group["spatial_mean"]["Ts"])
close(smoke)

## 5. Full simulation

The full window is 1 May to 3 October 2005 (3743 hourly steps). On a typical laptop this takes **several hours** (the MATLAB original used an 18-worker parallel pool for the same task). Start it when you do not need the machine for a while; the result streams to `exercise_2d_results.nc`, so you can also interrupt and keep what was written so far.

We store spatial maps of the time-average state at the MATLAB demo's snapshot steps: t = 40 (about two days, spin-up), t = 2160 (about three months) and the end of the run.

In [ ]:
# FULL RUN: several hours on a laptop.
# Comment the error below to enable the cell after reading the note above.
error("Uncomment the body of this cell to run the full simulation")

diagnostics, manager = run_simulation(
    model,
    forcing;
    options = options,
    run_options = SpatialRunOptions(
        output_level = AllOutputs(),
        output_filename = joinpath(data_dir, "exercise_2d_results.nc"),
        compress = true,
        parallel = false,
        tstore = [40, 2160],
    ),
);

## 6. Analyse the results

The cells below read `exercise_2d_results.nc` (produced by the full run). If you have not run the full simulation yet, they will fail with a missing-file error.

The output file mirrors the MATLAB demo's outputs:

| Group | Content | MATLAB analogue |
|---|---|---|
| `spatial_mean` / `spatial_std` | catchment-average time series and their standard deviation | `OUTPUT_..._AVG.dat`, `OUTPUT_..._STD.dat` |
| `point_1` | time series at the tracked grid point | `OUTPUT_..._PIXEL_1.dat` |
| `time_average` | spatial maps of time-averaged state at the snapshot steps | `OUTPUT_..._SPATIAL_t.mat` |
| `grid` | static grid data (DTM, land use, ...) | `GeoExample.mat` |

In [ ]:
results = NCDataset(joinpath(data_dir, "exercise_2d_results.nc"))
sm = results.group["spatial_mean"]
nothing

### 6.1 Catchment-average time series

In [ ]:
hours = 1:size(sm["Ts"], 1)
days = hours ./ 24

p1 = plot(days, Array(sm["Ts"]), label = "Ts", ylabel = "Surface temperature [°C]", legend = :best)
p2 = plot(days, Array(sm["QE"]), label = "QE", ylabel = "Latent heat [W/m²]", legend = :best)
p3 = plot(days, Array(sm["H"]), label = "H", ylabel = "Sensible heat [W/m²]", legend = :best)
p4 = plot(days, Array(sm["ZWT"]), label = "ZWT", ylabel = "Water table depth [mm]", legend = :best)
plot(p1, p2, p3, p4, layout = (4, 1), size = (800, 900), xlabel = "Time [days]")

**Question 6.1**: Relate the evaporation peaks in `QE` to rainfall events and to the seasonal trajectory of `Ts`. Which limitation dominates in early May, which in late summer?

**Question 6.2**: The water table depth (`ZWT`) responds to recharge. Does it rise during the May rains in your run? Compare with a neighbour: if your `Kbot` differs, how does the long-term `ZWT` change?

### 6.2 Soil moisture profile at the tracked point

In [ ]:
pt = results.group["point_1"]
V = Array(pt["V"]) # (time, layers)
heatmap(days, 1:size(V, 2), V'; xlabel = "Time [days]", ylabel = "Soil layer",
    colorbar_title = "Water content [‰]", title = "Soil moisture profile at point 1")

### 6.3 Spatial maps at the snapshot steps

In [ ]:
ta = results.group["time_average"]
Ts_maps = Array(ta["Ts"]) # (m_cell, n_cell, naver)

snapshots = ["t = 40 (spin-up)", "t = 2160 (~3 months)", "t = end"]
plots = [heatmap(Ts_maps[:, :, i]'; title = "Ts, $(snapshots[i])", clims = (0, 30), aspect_ratio = 1)
         for i in 1:min(3, size(Ts_maps, 3))]
plot(plots...; layout = (1, length(plots)), size = (300 * length(plots), 300))

In [ ]:
V_maps = Array(ta["V"]) # (m_cell, n_cell, layers, naver)
layer = 1
plots = [heatmap(V_maps[:, :, layer, i]'; title = "V (layer $layer), $(snapshots[i])", aspect_ratio = 1)
         for i in 1:min(3, size(V_maps, 4))]
plot(plots...; layout = (1, length(plots)), size = (300 * length(plots), 300))

**Question 6.3**: The maps inherit the catchment's topography. Where does the soil stay wettest, and how does the elevation gradient explain it?

**Question 6.4 (sensitivity)**: Change `Kbot` by a factor of 10 (e.g. to 1.1 mm/h), re-run (you can shorten the window in `create_data_2d.jl`), and compare `ZWT` and the maps. Which parts of the catchment respond most?

In [ ]:
close(results)